# 02 – Loading the Protected Data

The course data lives on LRZ Sync+Share behind a password-protected link. This notebook downloads it into `data/intern/`, a folder that is excluded from Git so the data never ends up on GitHub.

You will be asked for the password each time. It is not saved anywhere.

## 1. Setup

Only run this cell – you don't need to understand the details. It defines a function that logs in to the share and downloads its content.

In [ ]:
import getpass
import http.cookiejar
import io
import urllib.parse
import urllib.request
import zipfile
from html.parser import HTMLParser
from pathlib import Path

SHARE_LINK = "https://syncandshare.lrz.de/getlink/fiAeyMTuyiSWFG4b15dbPs/"

# Inside the course folder the data goes to data/intern/, elsewhere (e.g. Colab) to ./data/intern/
DATA_DIR = Path("../data/intern") if Path("../data").exists() else Path("data/intern")


class SharePage(HTMLParser):
    """Collects the attributes of all HTML elements that have an id."""

    def __init__(self):
        super().__init__()
        self.by_id = {}

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if "id" in attrs:
            self.by_id[attrs["id"]] = attrs


def download_share(link, password):
    """Logs in to a password-protected Sync+Share link and returns (filename, content)."""
    opener = urllib.request.build_opener(
        urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar())
    )

    form = urllib.parse.urlencode({"password": password}).encode()
    with opener.open(link, data=form) as response:
        page = SharePage()
        page.feed(response.read().decode("utf-8"))

    button = page.by_id.get("filelink_download")
    if not button or not button.get("href"):
        raise PermissionError("Could not open the share: wrong password, or the link has expired.")

    url = urllib.parse.urljoin(link, button["href"])
    token = page.by_id.get("fileLikeToken", {}).get("value")
    if token:
        url += ("&" if "?" in url else "?") + urllib.parse.urlencode({"fileLinkToken": token})

    with opener.open(url) as response:
        filename = response.headers.get_filename() or "download"
        return Path(filename).name, response.read()

## 2. Download

Run this cell and type the password into the box that appears. A shared folder arrives as a zip file and is unpacked automatically.

In [ ]:
filename, content = download_share(SHARE_LINK, getpass.getpass("Password: "))

DATA_DIR.mkdir(parents=True, exist_ok=True)

if zipfile.is_zipfile(io.BytesIO(content)):
    zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
else:
    (DATA_DIR / filename).write_bytes(content)

print(f"Downloaded {filename} ({len(content) / 1024:.0f} KB) into {DATA_DIR.resolve()}")

## 3. What did we get?

In [ ]:
for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(DATA_DIR)):50} {path.stat().st_size / 1024:8.0f} KB")